# Incident Response Runbook: Defense Evasion - Obfuscated Files or Information

**Tactic:** Defense Evasion
**Technique:** Obfuscated Files or Information (T1027)
**Severity:** HIGH

## Overview

This runbook provides step-by-step incident response procedures for Obfuscated Files or Information activities.

## Incident Response Phases

1. **Detection & Analysis**
2. **Containment**
3. **Eradication**
4. **Recovery**
5. **Post-Incident Activities**


## Phase 1: Detection & Analysis

### Objectives
- Confirm the incident
- Identify the scope
- Assess impact


In [ ]:
import json
import re
from datetime import datetime
import subprocess
import os
from splunk_data_collector import SplunkDataCollector
from crowdstrike_response import CrowdStrikeResponse
from iris_integration import create_iris_case, update_iris_case
from misp_integration import search_iocs, create_event
from shuffle_integration import trigger_workflow

# Step 1: Detection & Analysis
print("=" * 60)
print("STEP 1: Detection & Analysis")
print("=" * 60)

# Initialize security integrations
splunk = SplunkDataCollector()
crowdstrike = CrowdStrikeResponse()
alert_data = {
    'alert_id': 'ALERT-001',
    'timestamp': datetime.now().isoformat(),
    'tactic': 'Defense Evasion',
    'technique': 'Obfuscated Files or Information',
    'severity': 'MEDIUM'
}

print(f"Alert ID: {alert_data['alert_id']}")
print(f"Timestamp: {alert_data['timestamp']}")
print(f"Tactic: {alert_data['tactic']}")
print(f"Technique: {alert_data['technique']}")
print(f"Severity: {alert_data['severity']}")

# Query Splunk for obfuscation indicators
print(f"\n[ACTION] Querying Splunk for obfuscation activities...")
obfuscation_queries = [
    'index=* "base64" OR "xor" OR "rot13" OR "obfuscate" | stats count by host, user, file_path',
    'index=* source="file_monitoring" entropy>7.5 | where size < 1000000 | stats count by host, file_path, entropy',
    'index=* "powershell -e" OR "FromBase64String" OR "IEX" | stats count by host, user, command',
    'index=* "packer" OR "upx" OR "themida" OR "pecompact" | stats count by host, user, file_path',
    'index=* source="antivirus" signature="*obfuscated*" OR signature="*packed*" | stats count by host, file_path'
]

suspicious_activities = []
for query in obfuscation_queries:
    try:
        results = splunk.search(query, earliest_time="-24h@h", latest_time="now")
        if results and len(results) > 0:
            suspicious_activities.extend(results)
            print(f"   Found {len(results)} suspicious obfuscation events")
    except Exception as e:
        print(f"   Query failed: {e}")

# Analyze with CrowdStrike for obfuscated file behavior
print(f"\n[ACTION] Analyzing endpoint behavior with CrowdStrike...")
affected_systems = []
for activity in suspicious_activities:
    try:
        host_info = crowdstrike.get_host_info(activity.get('host', ''))
        if host_info:
            affected_systems.append(host_info)
            print(f"   Host {activity.get('host')} confirmed active")
    except Exception as e:
        print(f"   Failed to get host info: {e}")

# Create IRIS case for tracking
print(f"\n[ACTION] Creating IRIS case for incident tracking...")
try:
    iris_case = create_iris_case(
        title=f"Obfuscation Incident - {alert_data['alert_id']}",
        description=f"Detected obfuscated files or information affecting {len(affected_systems)} systems",
        severity="Medium",
        tags=["defense-evasion", "obfuscation", "malware", "T1027"]
    )
    iris_case_id = iris_case.get('case_id', 'IRIS-UNKNOWN')
    print(f"   IRIS case created: {iris_case_id}")
except Exception as e:
    print(f"   Failed to create IRIS case: {e}")
    iris_case_id = None

# Check MISP for related threat intelligence
print(f"\n[ACTION] Checking MISP for related threat intelligence...")
misp_indicators = []
for activity in suspicious_activities[:3]:
    try:
        iocs = search_iocs(activity.get('file_path', ''), type='filename')
        if iocs:
            misp_indicators.extend(iocs)
            print(f"   Found {len(iocs)} related IOCs in MISP")
    except Exception as e:
        print(f"   MISP search failed: {e}")

print(f"\n Detection complete:")
print(f"  - {len(suspicious_activities)} suspicious obfuscation activities detected")
print(f"  - {len(affected_systems)} systems affected")
print(f"  - {len(misp_indicators)} related threat intelligence indicators")


## Phase 2: Containment

### Objectives
- Stop the attack
- Isolate affected systems
- Prevent spread


In [ ]:
# Step 2: Containment
print("\n" + "=" * 60)
print("STEP 2: Containment")
print("=" * 60)

isolated_systems = []
quarantined_files = []
blocked_processes = []

# Isolate affected systems
print(f"\n[ACTION] Isolating affected systems...")
for system in affected_systems:
    try:
        isolation_result = crowdstrike.isolate_host(system['device_id'])
        if isolation_result.get('status') == 'success':
            isolated_systems.append(system['hostname'])
            print(f"   Isolated system: {system['hostname']}")
        else:
            print(f"   Failed to isolate {system['hostname']}: {isolation_result}")
    except Exception as e:
        print(f"   Isolation failed for {system.get('hostname', 'unknown')}: {e}")

# Quarantine obfuscated files
print(f"\n[ACTION] Quarantining obfuscated files...")
for activity in suspicious_activities:
    try:
        if activity.get('file_path'):
            file_quarantine = crowdstrike.quarantine_file(
                activity.get('host', ''),
                activity.get('file_path', '')
            )
            if file_quarantine.get('status') == 'success':
                quarantined_files.append(activity.get('file_path', ''))
                print(f"   Quarantined file: {activity.get('file_path', '')}")
    except Exception as e:
        print(f"   File quarantine failed: {e}")

# Block obfuscation processes
print(f"\n[ACTION] Blocking obfuscation processes...")
for activity in suspicious_activities:
    try:
        processes = crowdstrike.get_processes(activity.get('host', ''), activity.get('user', ''))
        for proc in processes:
            if any(keyword in proc.get('name', '').lower() for keyword in ['powershell', 'cmd', 'bash', 'sh', 'python', 'perl']):
                if any(obfus in proc.get('command_line', '').lower() for obfus in ['-e', 'frombase64string', 'iex', 'base64', 'xor']):
                    termination_result = crowdstrike.terminate_process(activity.get('host', ''), proc['pid'])
                    if termination_result.get('status') == 'success':
                        blocked_processes.append(f"{activity.get('host')}:{proc['name']}")
                        print(f"   Terminated obfuscation process: {proc['name']} on {activity.get('host')}")
    except Exception as e:
        print(f"   Process termination failed: {e}")

# Enable enhanced obfuscation detection
print(f"\n[ACTION] Enabling enhanced obfuscation detection...")
try:
    obfuscation_monitoring = trigger_workflow(
        workflow_name="enhanced_obfuscation_monitoring",
        parameters={
            'duration': '72h',
            'alert_on_high_entropy': True,
            'alert_on_packed_files': True,
            'alert_on_encoded_content': True,
            'target_systems': [sys['hostname'] for sys in affected_systems]
        }
    )
    if obfuscation_monitoring.get('status') == 'success':
        print(f"   Enhanced obfuscation monitoring enabled for {len(affected_systems)} systems")
    else:
        print(f"   Failed to enable obfuscation monitoring: {obfuscation_monitoring}")
except Exception as e:
    print(f"   Obfuscation monitoring setup failed: {e}")

print(f"\n Containment complete:")
print(f"  - {len(isolated_systems)} systems isolated")
print(f"  - {len(quarantined_files)} files quarantined")
print(f"  - {len(blocked_processes)} obfuscation processes terminated")
print(f"  - Enhanced obfuscation monitoring enabled")


## Phase 3: Eradication

### Objectives
- Remove malicious artifacts
- Close vulnerabilities
- Verify threat removal


In [ ]:
# Step 3: Eradication
print("\n" + "=" * 60)
print("STEP 3: Eradication")
print("=" * 60)

deobfuscated_files = []
removed_packers = []
cleaned_systems = []

# Analyze obfuscated files with threat intelligence
print(f"\n[ACTION] Analyzing obfuscated files with threat intelligence...")
obfuscation_analysis = {}
for activity in suspicious_activities:
    file_path = activity.get('file_path', '') if activity.get('file_path') else 'unknown'
    if file_path not in obfuscation_analysis:
        try:
            misp_analysis = search_iocs(file_path, type='filename')
            obfuscation_analysis[file_path] = {
                'threat_level': len(misp_analysis) if misp_analysis else 0,
                'indicators': misp_analysis[:3] if misp_analysis else [],
                'entropy': activity.get('entropy', 0)
            }
            print(f"   Analyzed file: {file_path} (threat level: {obfuscation_analysis[file_path]['threat_level']}, entropy: {obfuscation_analysis[file_path]['entropy']})")
        except Exception as e:
            print(f"   File analysis failed for {file_path}: {e}")

# Deobfuscate and analyze files
print(f"\n[ACTION] Deobfuscating and analyzing files...")
for activity in suspicious_activities:
    try:
        if activity.get('file_path'):
            deobfuscation_result = crowdstrike.deobfuscate_and_analyze(
                activity.get('host', ''),
                activity.get('file_path', '')
            )
            if deobfuscation_result.get('status') == 'analyzed':
                deobfuscated_files.append(activity.get('file_path', ''))
                print(f"   Deobfuscated and analyzed: {activity.get('file_path', '')}")
    except Exception as e:
        print(f"   Deobfuscation failed for {activity.get('file_path', '')}: {e}")

# Remove packers and obfuscation tools
print(f"\n[ACTION] Removing packers and obfuscation tools...")
for system in affected_systems:
    try:
        # Scan for and remove obfuscation tools
        scan_result = crowdstrike.scan_and_remove(
            system['device_id'],
            signatures=['packers', 'obfuscators', 'crypters', 'compressors']
        )
        if scan_result.get('removed_files', []):
            removed_packers.extend(scan_result['removed_files'])
            print(f"   Removed {len(scan_result['removed_files'])} obfuscation tools from {system['hostname']}")
    except Exception as e:
        print(f"   Packer removal failed for {system.get('hostname', 'unknown')}: {e}")

# Clean system of obfuscated content
print(f"\n[ACTION] Cleaning system of obfuscated content...")
for system in affected_systems:
    try:
        cleaning_result = crowdstrike.clean_obfuscated_content(system['device_id'])
        if cleaning_result.get('status') == 'cleaned':
            cleaned_systems.append(system['hostname'])
            print(f"   Cleaned obfuscated content from {system['hostname']}")
    except Exception as e:
        print(f"   System cleaning failed for {system.get('hostname', 'unknown')}: {e}")

# Verify threat removal
print(f"\n[ACTION] Verifying threat removal...")
verification_results = []
for system in affected_systems:
    try:
        verify_result = crowdstrike.verify_security_posture(system['device_id'])
        verification_results.append({
            'system': system['hostname'],
            'status': 'clean' if verify_result.get('threats_detected', 0) == 0 else 'threats_remaining',
            'threats': verify_result.get('threats_detected', 0)
        })
        status = "" if verify_result.get('threats_detected', 0) == 0 else ""
        print(f"  {status} Verification for {system['hostname']}: {verify_result.get('threats_detected', 0)} threats detected")
    except Exception as e:
        print(f"   Verification failed for {system.get('hostname', 'unknown')}: {e}")

print(f"\n Eradication complete:")
print(f"  - {len(deobfuscated_files)} files deobfuscated and analyzed")
print(f"  - {len(removed_packers)} packers and obfuscators removed")
print(f"  - {len(cleaned_systems)} systems cleaned")
print(f"  - {len([v for v in verification_results if v['status'] == 'clean'])} systems verified clean")


## Phase 4: Recovery

### Objectives
- Restore systems
- Validate functionality
- Monitor for issues


In [ ]:
# Step 4: Recovery
print("\n" + "=" * 60)
print("STEP 4: Recovery")
print("=" * 60)

restored_systems = []
reimaged_hosts = []
restored_services = []

# Restore affected systems
print(f"\n[ACTION] Restoring affected systems...")
for system in affected_systems:
    try:
        # Restore from clean backup if available
        restore_result = crowdstrike.restore_from_backup(
            system['device_id'],
            backup_type='clean'
        )
        if restore_result.get('status') == 'restored':
            restored_systems.append(system['hostname'])
            print(f"   Restored {system['hostname']} from clean backup")
        else:
            # If no clean backup, reimage the system
            reimage_result = crowdstrike.reimage_system(system['device_id'])
            if reimage_result.get('status') == 'reimaged':
                reimaged_hosts.append(system['hostname'])
                print(f"   Reimaged {system['hostname']} (no clean backup available)")
    except Exception as e:
        print(f"   System restoration failed for {system.get('hostname', 'unknown')}: {e}")

# Restore critical services
print(f"\n[ACTION] Restoring critical services...")
critical_services = ['antivirus', 'firewall', 'logging', 'monitoring']
for system in affected_systems:
    try:
        for service in critical_services:
            service_result = crowdstrike.restore_service(
                system['device_id'],
                service_name=service
            )
            if service_result.get('status') == 'restored':
                restored_services.append(f"{system['hostname']}:{service}")
                print(f"   Restored {service} service on {system['hostname']}")
    except Exception as e:
        print(f"   Service restoration failed for {system.get('hostname', 'unknown')}: {e}")

# Validate system integrity
print(f"\n[ACTION] Validating system integrity...")
integrity_checks = []
for system in affected_systems:
    try:
        integrity_result = crowdstrike.validate_system_integrity(system['device_id'])
        integrity_checks.append({
            'system': system['hostname'],
            'integrity': integrity_result.get('integrity_score', 0),
            'issues': integrity_result.get('issues_found', [])
        })
        status = "" if integrity_result.get('integrity_score', 0) > 80 else ""
        print(f"  {status} Integrity check for {system['hostname']}: {integrity_result.get('integrity_score', 0)}% ({len(integrity_result.get('issues_found', []))} issues)")
    except Exception as e:
        print(f"   Integrity validation failed for {system.get('hostname', 'unknown')}: {e}")

# Update security policies
print(f"\n[ACTION] Updating security policies to prevent obfuscation...")
policy_updates = []
try:
    # Enhance file monitoring for obfuscation patterns
    policy_result = crowdstrike.update_security_policy(
        policy_type='file_monitoring',
        rules={
            'obfuscation_detection': True,
            'entropy_analysis': True,
            'packer_detection': True,
            'compression_monitoring': True
        }
    )
    if policy_result.get('status') == 'updated':
        policy_updates.append('file_monitoring')
        print(f"   Updated file monitoring policy for obfuscation detection")

    # Update behavioral analytics
    behavior_result = crowdstrike.update_security_policy(
        policy_type='behavioral_analytics',
        rules={
            'anomaly_detection': True,
            'obfuscation_patterns': True,
            'suspicious_execution': True
        }
    )
    if behavior_result.get('status') == 'updated':
        policy_updates.append('behavioral_analytics')
        print(f"   Updated behavioral analytics policy")
except Exception as e:
    print(f"   Policy update failed: {e}")

print(f"\n Recovery complete:")
print(f"  - {len(restored_systems)} systems restored from backup")
print(f"  - {len(reimaged_hosts)} systems reimaged")
print(f"  - {len(restored_services)} services restored")
print(f"  - {len(policy_updates)} security policies updated")


## Phase 5: Post-Incident Activities

### Objectives
- Document findings
- Implement improvements
- Share lessons learned


In [ ]:
# Step 5: Post-Incident
print("\n" + "=" * 60)
print("STEP 5: Post-Incident")
print("=" * 60)

# Document incident details
incident_summary = {
    'incident_id': alert_data.get('incident_id', 'unknown'),
    'technique': 'Defense Evasion - Obfuscated Files or Information',
    'detection_time': alert_data.get('timestamp', 'unknown'),
    'affected_systems': len(affected_systems),
    'obfuscated_files_detected': len(suspicious_activities),
    'containment_actions': len(isolated_hosts) + len(quarantined_files) + len(terminated_processes),
    'eradication_actions': len(deobfuscated_files) + len(removed_packers) + len(cleaned_systems),
    'recovery_actions': len(restored_systems) + len(reimaged_hosts) + len(restored_services),
    'timeline': {
        'detection': alert_data.get('timestamp', 'unknown'),
        'containment': 'completed',
        'eradication': 'completed',
        'recovery': 'completed',
        'closure': 'now'
    }
}

# Create comprehensive incident report
print(f"\n[ACTION] Creating comprehensive incident report...")
try:
    iris_report = iris.create_incident_report(
        incident_id=alert_data.get('incident_id', 'unknown'),
        title=f"Defense Evasion - Obfuscated Files Incident {alert_data.get('incident_id', 'unknown')}",
        description=f"Detected and responded to obfuscated files and information on {len(affected_systems)} systems. {len(suspicious_activities)} suspicious files were identified and processed.",
        severity='high',
        status='closed',
        details={
            'technique': 'T1027 - Obfuscated Files or Information',
            'indicators': obfuscation_analysis,
            'affected_assets': [s['hostname'] for s in affected_systems],
            'response_actions': {
                'detection': f"Splunk queries identified {len(suspicious_activities)} suspicious activities",
                'containment': f"Isolated {len(isolated_hosts)} hosts, quarantined {len(quarantined_files)} files, terminated {len(terminated_processes)} processes",
                'eradication': f"Deobfuscated {len(deobfuscated_files)} files, removed {len(removed_packers)} packers, cleaned {len(cleaned_systems)} systems",
                'recovery': f"Restored {len(restored_systems)} systems, reimaged {len(reimaged_hosts)} hosts, restored {len(restored_services)} services"
            },
            'threat_intelligence': {
                'misp_iocs': sum(len(analysis.get('indicators', [])) for analysis in obfuscation_analysis.values()),
                'threat_levels': [analysis.get('threat_level', 0) for analysis in obfuscation_analysis.values()]
            }
        }
    )
    print(f"   Incident report created in IRIS: {iris_report.get('report_id', 'unknown')}")
except Exception as e:
    print(f"   Report creation failed: {e}")

# Share indicators with threat intelligence community
print(f"\n[ACTION] Sharing indicators with threat intelligence community...")
try:
    shared_iocs = []
    for file_path, analysis in obfuscation_analysis.items():
        if analysis.get('threat_level', 0) > 0:
            misp_event = misp.create_event(
                title=f"Obfuscated File Detection: {file_path}",
                description=f"High-entropy obfuscated file detected during incident response. Entropy: {analysis.get('entropy', 0)}",
                attributes=[
                    {
                        'type': 'filename',
                        'value': file_path,
                        'comment': f'Detected in obfuscation incident {alert_data.get("incident_id", "unknown")}'
                    }
                ] + [
                    {
                        'type': 'sha256',
                        'value': indicator.get('hash', ''),
                        'comment': 'File hash from obfuscation analysis'
                    } for indicator in analysis.get('indicators', [])
                ]
            )
            if misp_event.get('status') == 'created':
                shared_iocs.append(file_path)
                print(f"   Shared IOC for {file_path} with MISP community")

    print(f"   Shared {len(shared_iocs)} indicators with threat intelligence community")
except Exception as e:
    print(f"   IOC sharing failed: {e}")

# Generate lessons learned
print(f"\n[ACTION] Generating lessons learned...")
lessons_learned = {
    'incident_type': 'Defense Evasion - Obfuscated Files',
    'key_findings': [
        f"Obfuscation techniques detected: {len([a for a in obfuscation_analysis.values() if a.get('threat_level', 0) > 0])} high-threat files",
        f"Average file entropy: {sum(a.get('entropy', 0) for a in obfuscation_analysis.values()) / len(obfuscation_analysis) if obfuscation_analysis else 0:.2f}",
        f"Systems affected: {len(affected_systems)}",
        f"Response time: Contained within {len(isolated_hosts)} minutes of detection"
    ],
    'improvements_needed': [
        'Enhance entropy-based file monitoring',
        'Implement real-time obfuscation detection',
        'Improve automated deobfuscation capabilities',
        'Strengthen backup and recovery procedures'
    ],
    'preventive_measures': [
        'Deploy advanced file integrity monitoring',
        'Implement behavioral analytics for obfuscation patterns',
        'Regular security policy updates for emerging threats',
        'Enhanced threat intelligence integration'
    ]
}

try:
    iris.add_lessons_learned(
        incident_id=alert_data.get('incident_id', 'unknown'),
        lessons=lessons_learned
    )
    print(f"   Lessons learned documented in IRIS")
except Exception as e:
    print(f"   Lessons learned documentation failed: {e}")

# Final status update
print(f"\n Incident {alert_data.get('incident_id', 'unknown')} successfully resolved:")
print(f"  - Technique: Defense Evasion - Obfuscated Files or Information (T1027)")
print(f"  - Status: Closed")
print(f"  - Systems Recovered: {len(restored_systems) + len(reimaged_hosts)}")
print(f"  - Threat Indicators Shared: {len(shared_iocs) if 'shared_iocs' in locals() else 0}")
print(f"  - Incident Report Generated: Yes")
print(f"  - Lessons Learned Documented: Yes")

print(f"\n{'=' * 60}")
print("INCIDENT RESPONSE COMPLETE")
print(f"{'=' * 60}")


## Summary

This incident response runbook has guided you through the complete lifecycle of responding to this incident.

### Key Takeaways
- Rapid detection and response are critical
- Containment prevents further damage
- Thorough eradication prevents reinfection
- Continuous monitoring ensures recovery

### Next Steps
- Review and approve the incident report
- Implement recommended preventive measures
- Schedule follow-up review
- Share lessons learned with the team
